In [0]:
!pip install openpyxl

DEPRECATION: Using the pkg_resources metadata backend is deprecated. pip 26.3 will enforce this behaviour change. A possible replacement is to use the default importlib.metadata backend, by unsetting the _PIP_USE_IMPORTLIB_METADATA environment variable. Discussion can be found at https://github.com/pypa/pip/issues/13317
Looking in indexes: https://pypi.org/simple, https://****@pkgs.dev.azure.com/EHI-DataEngineeringPlatform/_packaging/EHI-DataEngineeringPlatform/pypi/simple/
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
schemas = [
    "ody_tb",
    "ecars_tb"

]

In [0]:
# %python
 
# table_name = 'appl_cntrl_log_l1'
# env="edl_dev"
 
# for i in schemas:
#     print(f'creating backup of : {i}.appl_cntrl_log_l1')
#     spark.sql(f"""CREATE OR REPLACE TABLE t30day_tb.{i}_appl_cntrl_log_l1_t30backup_{env}
#         AS SELECT * FROM {env}.{i}.{table_name}
#         """
#     )
   

creating backup of : ody_tb.appl_cntrl_log_l1
creating backup of : reference_tb.appl_cntrl_log_l1
creating backup of : ace_gdpr_tb.appl_cntrl_log_l1
creating backup of : app_security_tb.appl_cntrl_log_l1
creating backup of : arms_tb.appl_cntrl_log_l1
creating backup of : arms_auto_tb.appl_cntrl_log_l1
creating backup of : cam_tb.appl_cntrl_log_l1
creating backup of : cntct_ctr_tb.appl_cntrl_log_l1
creating backup of : csdp_tb.appl_cntrl_log_l1
creating backup of : dcls_veh_resdbuy_tb.appl_cntrl_log_l1
creating backup of : dss_metrics_tb.appl_cntrl_log_l1
creating backup of : ecars_tb.appl_cntrl_log_l1
creating backup of : ecir_tb.appl_cntrl_log_l1
creating backup of : fltsht_tb.appl_cntrl_log_l1
creating backup of : indiv_pref_tb.appl_cntrl_log_l1
creating backup of : lrd_tb.appl_cntrl_log_l1
creating backup of : osdwadmin_tb.appl_cntrl_log_l1
creating backup of : paymt_tb.appl_cntrl_log_l1
creating backup of : pbta_tb.appl_cntrl_log_l1
creating backup of : prism_tb.appl_cntrl_log_l1
c

In [0]:
from pyspark.sql.functions import lit
from functools import reduce

catalog = "edl_dev"
df_list = []

for schema in schemas:
    try:
        df = (
            spark.table(f"{catalog}.{schema}.appl_cntrl_log_l1")
            .withColumn("catalog_schema", lit(schema))   # 👈 add this line
        )
        df_list.append(df)
    except:
        print(f"Skipping {schema}")

final_df = reduce(lambda x, y: x.unionByName(y), df_list)

# Replace literal "null" strings with actual nulls
clean_df = final_df.replace("null", None)

clean_df.createOrReplaceTempView("dev_df")

spark.sql("""
CREATE OR REPLACE TABLE t30day_tb.appl_cntrl_log_l1_t30backup_dev
USING DELTA
PARTITIONED BY (appl_cntrl_id)
AS SELECT * FROM dev_df
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
from pyspark.sql.functions import lit
from functools import reduce

catalog = "edl_prd"
df_list = []

for schema in schemas:
    try:
        df = (
            spark.table(f"{catalog}.{schema}.appl_cntrl_log_l1")
            .withColumn("catalog_schema", lit(schema))   # 👈 important
        )
        df_list.append(df)
    except:
        print(f"Skipping {schema}")

# Union all schemas
final_df = reduce(lambda x, y: x.unionByName(y), df_list)

# Replace "null" string with actual null
clean_df = final_df.replace("null", None)

# Convert to pandas
pdf = clean_df.toPandas()

# Save to Excel
pdf.to_excel("prd_full.xlsx", index=False)

In [0]:
# =========================================
# Read Excel -> Create Spark DF -> Create Table with Fixed Metadata -> Load Data
# =========================================

import pandas as pd

# 1. Read Excel using pandas
prd_df1 = pd.read_excel("prd_full.xlsx")

# 2. Convert pandas DataFrame to Spark DataFrame
prd_df = spark.createDataFrame(prd_df1)

# 3. Replace string 'null' with actual null
clean_df = prd_df.replace("null", None)

# 4. Create temp view
clean_df.createOrReplaceTempView("prd_df")

# 5. Create NEW table using fixed metadata (NO CTAS)
spark.sql("""
CREATE OR REPLACE TABLE t30day_tb.appl_cntrl_log_l1_t30backup_prd (
  appl_cntrl_id INT,
  appl_cntrl_grp VARCHAR(256),
  appl_cntrl_nam VARCHAR(256),
  run_set SMALLINT,
  run_set_seq SMALLINT,
  load_nam VARCHAR(256),
  load_tool VARCHAR(25),
  versioned VARCHAR(1),
  trunc_reload VARCHAR(1),
  src_typ VARCHAR(256),
  src_sys VARCHAR(256),
  src_obj VARCHAR(256),
  src_fld_list_txt VARCHAR(8000),
  src_row_cnt BIGINT,
  stg_typ VARCHAR(256),
  stg_sys VARCHAR(256),
  stg_obj VARCHAR(256),
  stg_fld_list_txt VARCHAR(8000),
  stg_row_cnt BIGINT,
  tgt_typ VARCHAR(256),
  tgt_sys VARCHAR(256),
  tgt_obj VARCHAR(256),
  tgt_fld_list_txt VARCHAR(8000),
  tgt_row_cnt BIGINT,
  wmk_fld VARCHAR(256),
  data_as_of_tsp TIMESTAMP,
  param_strt_tsp TIMESTAMP,
  param_end_tsp TIMESTAMP,
  sql_str1 VARCHAR(8000),
  sql_str2 VARCHAR(8000),
  sql_str3 VARCHAR(8000),
  sql_str_lst_exectd VARCHAR(8000),
  prmry_key VARCHAR(256),
  order_by_fld VARCHAR(256),
  part_key VARCHAR(256),
  appl_strt_tsp TIMESTAMP,
  appl_end_tsp TIMESTAMP,
  status_cde VARCHAR(10),
  err_cde VARCHAR(8000),
  err_msg VARCHAR(8000),
  actv_flg INT,
  note_txt VARCHAR(8000),
  config_ext STRING,
  criticality INT,
  sla INT,
  hard_delete INT,
  domain STRING,
  dl_upd_tsp TIMESTAMP,
  dl_crte_tsp TIMESTAMP,
  catalog_schema STRING
)
USING DELTA
PARTITIONED BY (appl_cntrl_id)
TBLPROPERTIES (
  'delta.minReaderVersion' = '1',
  'delta.minWriterVersion' = '2'
)
""")

# 6. Load data into the table (uses metadata as-is)
spark.sql("""
INSERT INTO t30day_tb.appl_cntrl_log_l1_t30backup_prd
SELECT * FROM prd_df
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql
select * from t30day_tb.appl_cntrl_log_l1_t30backup_prd limit 2

appl_cntrl_id,appl_cntrl_grp,appl_cntrl_nam,run_set,run_set_seq,load_nam,load_tool,versioned,trunc_reload,src_typ,src_sys,src_obj,src_fld_list_txt,src_row_cnt,stg_typ,stg_sys,stg_obj,stg_fld_list_txt,stg_row_cnt,tgt_typ,tgt_sys,tgt_obj,tgt_fld_list_txt,tgt_row_cnt,wmk_fld,data_as_of_tsp,param_strt_tsp,param_end_tsp,sql_str1,sql_str2,sql_str3,sql_str_lst_exectd,prmry_key,order_by_fld,part_key,appl_strt_tsp,appl_end_tsp,status_cde,err_cde,err_msg,actv_flg,note_txt,config_ext,criticality,sla,hard_delete,domain,dl_upd_tsp,dl_crte_tsp
300,EDL L1,ENTERPRISE DELTA LAKE L1 LOAD,0,0,l1_deltamerge_adls_ody_3,DATABRICKS NOTEBOOK,Y,N,QLIK ADLS,ORACLE,"ODY/{{CDC,FL,PURGE}}/{{GWYDB.FIR_OPT_COVERAGES_DTL,GWYDB.FIR_OPT_COVERAGES_DTL__ct}}/",null,0,GEN2,stqlikrntl1prdusc1,capture,null,0,DELTA,ody_tb,fir_opt_coverages_dtl,null,0,null,2025-10-02T05:14:55Z,null,2026-04-02T13:15:23Z,null,null,null,null,null,null,null,2026-04-02T13:25:27.057Z,2026-04-02T13:27:54.341Z,COMPLETE,null,SIDELOAD/RENAME COMPLETE,1,**ROLLED BACK on 2025-10-30**|**DECOMMISSIONED on 2025-07-11**,"{""max_parallel"": ""5"",""intervaloverride"": """",""timeout duration"": ""60000"",""appl_cntrl_override"": """",""checkpoint_storage_location"": ""stedlcheckpointprdusc1"",""checkpoint_storage_file_postfix"": ""_01"",""purge_start_time"": ""2300000"",""file_format"": ""parquet"",""file_compressed"": ""false"",""enable_worktable"": ""true"",""checkpoint_worktable_file_postfix"": ""_01""}",2,360,0,Rental,2026-04-02T13:27:54.341Z,2022-08-31T21:30:23.503Z
417,EDL L1,ENTERPRISE DELTA LAKE L1 WT,30,0,l1-deltaworktable-stcaptody03prdusc1,DATABRICKS NOTEBOOK,null,null,QLIK EVENT HUB,eh-cdc-ody-03-prd-usc1,eventhub=gwydb.rate_details_a,null,0,GEN2,stcaptody03prdusc1,capture,null,0,DELTA,ody_wt,null,null,0,null,2024-08-29T16:49:28Z,null,2024-10-11T14:03:22.729Z,null,null,null,null,null,null,null,2024-10-11T14:03:22.729Z,2024-10-11T14:03:38.165Z,COMPLETE,null,null,1,null,null,null,null,0,null,2024-10-11T14:03:38.165Z,2023-03-29T15:46:33.716Z


In [0]:
%sql
CREATE OR REPLACE TABLE t30day_tb.final_table 
USING DELTA 
PARTITIONED BY (appl_cntrl_id) AS

SELECT
    P.*,
    CASE 
        WHEN p.tgt_sys IS NOT NULL AND d.tgt_sys IS NULL THEN 'MISSING_IN_DEV'
        WHEN p.tgt_sys IS NULL AND d.tgt_sys IS NOT NULL THEN 'EXTRA_IN_DEV'
        WHEN p.load_nam <> d.load_nam THEN 'WORKFLOW_MISMATCH'
        ELSE 'MATCH'
    END AS status

FROM prd_df p
FULL OUTER JOIN dev_df d
    ON p.catalog_schema = d.catalog_schema
    AND p.tgt_sys = d.tgt_sys
    AND p.appl_cntrl_id = d.appl_cntrl_id

WHERE NOT (
        p.tgt_sys IS NOT NULL 
    AND d.tgt_sys IS NOT NULL 
    AND p.load_nam = d.load_nam
);

num_affected_rows,num_inserted_rows


In [0]:
dev_catalog = "edl_dev"

# Only required columns
mismatch_df = (
    spark.table("t30day_tb.final_table")
    .filter("status = 'WORKFLOW_MISMATCH'")
    .select("catalog_schema", "appl_cntrl_id", "tgt_sys", "load_nam")
)

schemas = [row.catalog_schema for row in mismatch_df.select("catalog_schema").distinct().collect()]

for schema in schemas:
    temp_view = f"mismatch_{schema}"

    mismatch_df.filter(f"catalog_schema = '{schema}'").createOrReplaceTempView(temp_view)

    # STEP 1 + 2 combined (efficient)
    spark.sql(f"""
        MERGE INTO {dev_catalog}.{schema}.appl_cntrl_log_l1 AS tgt
        USING {temp_view} AS src
        ON tgt.appl_cntrl_id = src.appl_cntrl_id
           AND tgt.tgt_sys = src.tgt_sys

        WHEN MATCHED THEN UPDATE SET
            tgt.actv_flg = 0,
            tgt.load_nam = src.load_nam
    """)

    print(f"[UPDATED] {schema} completed")

In [0]:
backup_table = "t30day_tb.appl_cntrl_log_l1_t30backup_dev"
dev_catalog = "edl_dev"

schemas = [row.catalog_schema for row in mismatch_df.select("catalog_schema").distinct().collect()]

for schema in schemas:
    spark.sql(f"""
        MERGE INTO {dev_catalog}.{schema}.appl_cntrl_log_l1 AS tgt
        USING (
            SELECT appl_cntrl_id, tgt_sys, actv_flg
            FROM {backup_table}
            WHERE catalog_schema = '{schema}'
        ) AS bkp
        ON tgt.appl_cntrl_id = bkp.appl_cntrl_id
           AND tgt.tgt_sys = bkp.tgt_sys

        WHEN MATCHED THEN UPDATE SET
            tgt.actv_flg = bkp.actv_flg
    """)

    print(f"[RESTORED] {schema} completed")

In [0]:
from pyspark.sql import functions as F

dev_catalog = "edl_dev"

# Only missing records
missing_df = (
    spark.table("t30day_tb.final_table")
    .filter("status = 'MISSING_IN_DEV'")
    .select("catalog_schema", "appl_cntrl_id", "tgt_sys")
)

prd_df = spark.table("t30day_tb.appl_cntrl_log_l1_t30backup_prd")

schemas = [row.catalog_schema for row in missing_df.select("catalog_schema").distinct().collect()]

for schema in schemas:
    # Get only relevant rows from PRD
    insert_df = (
        prd_df.alias("p")
        .join(
            missing_df.filter(f"catalog_schema = '{schema}'").alias("m"),
            on=["appl_cntrl_id", "tgt_sys"],
            how="inner"
        )
        .filter(f"p.catalog_schema = '{schema}'")
        .drop("catalog_schema")   # drop before insert (target won't have it)
    )

    target_table = f"{dev_catalog}.{schema}.appl_cntrl_log_l1"
    target_df = spark.table(target_table)

    # Match column order
    insert_df = insert_df.select(*target_df.columns)

    # Force actv_flg = 0
    insert_df = insert_df.withColumn("actv_flg", F.lit(0))

    # Overwrite timestamps
    timestamp_cols = [f.name for f in target_df.schema.fields if "timestamp" in str(f.dataType).lower()]
    for col in timestamp_cols:
        insert_df = insert_df.withColumn(col, F.current_timestamp())

    # Bulk insert
    insert_df.write.mode("append").insertInto(target_table)

    print(f"[INSERTED] {schema} completed")

[RESET] actv_flg=0 → ody_tb - 447
[INSERTED] ody_tb - 447
[RESET] actv_flg=0 → ody_tb - 449
[INSERTED] ody_tb - 449
[RESET] actv_flg=0 → ody_tb - 448
[INSERTED] ody_tb - 448
[RESET] actv_flg=0 → ody_tb - 450
[INSERTED] ody_tb - 450
[RESET] actv_flg=0 → ody_tb - 454
[INSERTED] ody_tb - 454
[RESET] actv_flg=0 → ody_tb - 451
[INSERTED] ody_tb - 451
[RESET] actv_flg=0 → ody_tb - 452
[INSERTED] ody_tb - 452
[RESET] actv_flg=0 → ody_tb - 453
[INSERTED] ody_tb - 453
[RESET] actv_flg=0 → ody_tb - 455
[INSERTED] ody_tb - 455
[RESET] actv_flg=0 → ody_tb - 459
[INSERTED] ody_tb - 459
[RESET] actv_flg=0 → ody_tb - 456
[INSERTED] ody_tb - 456
[RESET] actv_flg=0 → ody_tb - 460
[INSERTED] ody_tb - 460
[RESET] actv_flg=0 → ody_tb - 457
[INSERTED] ody_tb - 457
[RESET] actv_flg=0 → ody_tb - 458
[INSERTED] ody_tb - 458
[RESET] actv_flg=0 → ody_tb - 465
[INSERTED] ody_tb - 465
[RESET] actv_flg=0 → ody_tb - 461
[INSERTED] ody_tb - 461
[RESET] actv_flg=0 → ody_tb - 463
[INSERTED] ody_tb - 463
[RESET] actv_f

In [0]:
dev_catalog = "edl_dev"

schemas = [row.catalog_schema for row in missing_df.select("catalog_schema").distinct().collect()]

for schema in schemas:
    spark.sql(f"""
        MERGE INTO {dev_catalog}.{schema}.appl_cntrl_log_l1 AS tgt
        USING (
            SELECT appl_cntrl_id, tgt_sys, actv_flg
            FROM t30day_tb.appl_cntrl_log_l1_t30backup_prd
            WHERE catalog_schema = '{schema}'
        ) AS src
        ON tgt.appl_cntrl_id = src.appl_cntrl_id
           AND tgt.tgt_sys = src.tgt_sys

        WHEN MATCHED THEN UPDATE SET
            tgt.actv_flg = src.actv_flg
    """)

    print(f"[RESTORED] {schema} completed")

[RESTORED] actv_flg=1 → ody_tb - 459
[RESTORED] actv_flg=1 → ody_tb - 463
[RESTORED] actv_flg=1 → ody_tb - 457
[RESTORED] actv_flg=1 → ody_tb - 456
[RESTORED] actv_flg=1 → ody_tb - 458
[RESTORED] actv_flg=1 → ody_tb - 465
[RESTORED] actv_flg=1 → ody_tb - 455
[RESTORED] actv_flg=1 → ody_tb - 462
[RESTORED] actv_flg=1 → ody_tb - 460
[RESTORED] actv_flg=1 → ody_tb - 461
[RESTORED] actv_flg=1 → ody_tb - 464
[RESTORED] actv_flg=1 → ody_tb - 453
[RESTORED] actv_flg=1 → ody_tb - 449
[RESTORED] actv_flg=1 → ody_tb - 450
[RESTORED] actv_flg=1 → ody_tb - 448
[RESTORED] actv_flg=1 → ody_tb - 454
[RESTORED] actv_flg=1 → ody_tb - 452
[RESTORED] actv_flg=1 → ody_tb - 447
[RESTORED] actv_flg=1 → ody_tb - 451
